In [1]:
from pathlib import Path
import pyreadstat
import pandas as pd
import numpy as np

# Resolve repository root dynamically whether launched from root or notebooks/
cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
DATA_DIR = PROJECT_ROOT / "data"

# Paths to official PISA 2022 (CY08) SAS datasets
school_file_path = (
    DATA_DIR
    / "CY09_MS_SCH_PUF"
    / "SCH_QQQ_SAS"
    / "CY08MSP_SCH_QQQ.SAS7BDAT"
)

student_file_path = (
    DATA_DIR
    / "STU_QQQ_SAS"
    / "CY08MSP_STU_QQQ.SAS7BDAT"
)

assert school_file_path.exists(), f"School file missing at: {school_file_path}"
assert student_file_path.exists(), f"Student file missing at: {student_file_path}"

print("All dependencies imported and PISA 2022 raw file paths verified successfully!")
print("Project root:", PROJECT_ROOT)

All dependencies imported and PISA 2022 raw file paths verified successfully!
Project root: C:\Users\ASUS\pisa_project


In [2]:
# Define targeted school-level policy, infrastructure, and staffing indicators
school_target_cols = [
    'CNT', 'CNTSCHID',         # Composite primary keys (Country code and School ID)
    'CLSIZE',                  # Average class size
    'STRATIO',                 # Student-to-teacher ratio
    'PROATCE',                 # Proportion of fully certified teachers
    'EDUSHORT',                # Shortage of educational material and infrastructure
    'STAFFSHORT',              # Shortage of instructional staff
    'SCHLTYPE'                 # School governance type (public vs. private)
]

# Read designated columns from the 2022 school questionnaire file
df_school, school_meta = pyreadstat.read_sas7bdat(
    school_file_path, 
    usecols=school_target_cols
)

print(f"School data loaded: {df_school.shape[0]} schools across {df_school.shape[1]} attributes.")
df_school.head(3)

School data loaded: 21629 schools across 8 attributes.


,CNT,CNTSCHID,SCHLTYPE,PROATCE,STRATIO,STAFFSHORT,EDUSHORT,CLSIZE
0,ALB,800001.0,3.0,1.0,16.9351,-0.2968,1.2048,33.0
1,ALB,800002.0,3.0,1.0,11.4375,-1.4551,2.9595,13.0
2,ALB,800003.0,1.0,1.0,7.1176,0.2833,0.6299,13.0


In [3]:
# Retain all 10 plausible values (PV1-PV10) and all 80 Fay's BRR replicate weights (W_FSTURWT1-80)
# Replicate weights are required for unbiased standard error estimation in two-stage stratified samples
pv_cols = [f'PV{i}MATH' for i in range(1, 11)]
rep_weight_cols = [f'W_FSTURWT{i}' for i in range(1, 81)]

student_target_cols = [
    'CNT', 'CNTSCHID',         # Composite relational keys matching school data
    'ESCS',                    # Index of economic, social, and cultural status
    'W_FSTUWT',                # Final student survey weight
] + pv_cols + rep_weight_cols

# Stream selected attributes from the student SAS dataset
df_student, student_meta = pyreadstat.read_sas7bdat(
    student_file_path, 
    usecols=student_target_cols
)

print(f"Student data loaded: {df_student.shape[0]} students across {df_student.shape[1]} attributes.")
df_student[['CNT', 'CNTSCHID', 'ESCS', 'W_FSTUWT', 'PV1MATH', 'PV10MATH', 'W_FSTURWT1', 'W_FSTURWT80']].head(3)

Student data loaded: 613744 students across 94 attributes.


,CNT,CNTSCHID,ESCS,W_FSTUWT,PV1MATH,PV10MATH,W_FSTURWT1,W_FSTURWT80
0,ALB,800282.0,1.1112,3.15874,179.583,254.745,4.74716,4.70887
1,ALB,800115.0,-3.0507,4.34524,308.247,329.659,6.48246,6.40402
2,ALB,800242.0,-0.1867,7.83860,267.514,295.401,11.93474,3.96558


In [4]:
# Select expanded European cohort (note: Ukrainian regions are coded as 'QUR' in PISA 2022)
target_countries = [
    'DEU', 'FRA', 'POL', 'EST', 'ITA', 'ESP', 
    'AUT', 'BEL', 'CZE', 'FIN', 'NLD', 'PRT', 'SWE', 'QUR'
]

df_school_sub = df_school[df_school['CNT'].isin(target_countries)].copy()
df_student_sub = df_student[df_student['CNT'].isin(target_countries)].copy()

# Harmonize key data types
df_student_sub['CNTSCHID'] = df_student_sub['CNTSCHID'].astype(float)
df_school_sub['CNTSCHID'] = df_school_sub['CNTSCHID'].astype(float)
df_student_sub['CNT'] = df_student_sub['CNT'].astype(str)
df_school_sub['CNT'] = df_school_sub['CNT'].astype(str)

# 1. Compute school-average ESCS (SCH_ESCS) BEFORE student-level imputation (critical for RQ2 peer effects)
school_escs = (
    df_student_sub.groupby(['CNT', 'CNTSCHID'])['ESCS']
    .mean()
    .rename('SCH_ESCS')
    .reset_index()
)
df_school_sub = pd.merge(df_school_sub, school_escs, on=['CNT', 'CNTSCHID'], how='left')

# 2. Impute school-level numeric covariates within country + fallback to pooled median if country is 100% NaN
school_impute_candidates = ['STRATIO', 'PROATCE', 'EDUSHORT', 'STAFFSHORT', 'CLSIZE', 'SCH_ESCS']
school_impute_cols = [col for col in school_impute_candidates if col in df_school_sub.columns]

print("School-level missing values BEFORE imputation:")
print(df_school_sub[school_impute_cols].isna().sum())

for col in school_impute_cols:
    df_school_sub[f'{col}_imputed'] = df_school_sub[col].isna().astype(int)
    # Primary: country-level median
    df_school_sub[col] = df_school_sub.groupby('CNT')[col].transform(
        lambda group: group.fillna(group.median())
    )
    # Fallback: if an entire country did not administer the question (e.g. AUT), use pooled European median
    df_school_sub[col] = df_school_sub[col].fillna(df_school_sub[col].median())

print("\nSchool-level missing values AFTER imputation:")
print(df_school_sub[school_impute_cols].isna().sum())

# Impute categorical SCHLTYPE using country-level mode + pooled mode fallback
if 'SCHLTYPE' in df_school_sub.columns:
    df_school_sub['SCHLTYPE_imputed'] = df_school_sub['SCHLTYPE'].isna().astype(int)
    df_school_sub['SCHLTYPE'] = df_school_sub.groupby('CNT')['SCHLTYPE'].transform(
        lambda x: x.fillna(x.mode()[0] if not x.mode().empty else np.nan)
    )
    df_school_sub['SCHLTYPE'] = df_school_sub['SCHLTYPE'].fillna(df_school_sub['SCHLTYPE'].mode()[0])

# 3. Student-level ESCS: create missingness flag BEFORE country-median imputation
print("\nStudent ESCS missing values BEFORE imputation:", df_student_sub['ESCS'].isna().sum())
df_student_sub['ESCS_imputed'] = df_student_sub['ESCS'].isna().astype(int)
df_student_sub['ESCS'] = df_student_sub.groupby('CNT')['ESCS'].transform(
    lambda group: group.fillna(group.median())
)
df_student_sub['ESCS'] = df_student_sub['ESCS'].fillna(df_student_sub['ESCS'].median())
print("Student ESCS missing values AFTER imputation:", df_student_sub['ESCS'].isna().sum())

# 4. Validate uniqueness of school keys and execute many-to-one inner merge
assert not df_school_sub.duplicated(subset=['CNT', 'CNTSCHID']).any(), "Duplicate school keys found!"

before_by_country = df_student_sub.groupby('CNT').size().rename('before_merge')

merged_df = pd.merge(
    df_student_sub,
    df_school_sub,
    on=['CNT', 'CNTSCHID'],
    how='inner',
    validate='many_to_one'
)

after_by_country = merged_df.groupby('CNT').size().rename('after_merge')

# 5. Compute Normalized Senate Weights (SENWT) so each country sums to 1,000
merged_df['SENWT'] = merged_df.groupby('CNT')['W_FSTUWT'].transform(
    lambda w: w / w.sum() * 1000.0
)

# Summarize merge retention and weight balance
merge_retention = pd.concat([before_by_country, after_by_country], axis=1).fillna(0).astype(int)
merge_retention['retained_pct'] = (100 * merge_retention['after_merge'] / merge_retention['before_merge']).round(2)
merge_retention['sample_share_%'] = (100 * merge_retention['after_merge'] / len(merged_df)).round(2)
merge_retention['senwt_sum'] = merged_df.groupby('CNT')['SENWT'].sum().round(1)

print(f"\nMerged dimensions: {merged_df.shape[0]} students across {merged_df['CNT'].nunique()} countries.")
display(merge_retention)

School-level missing values BEFORE imputation:
STRATIO       535
PROATCE       669
EDUSHORT      397
STAFFSHORT    834
CLSIZE        406
SCH_ESCS       10
dtype: int64

School-level missing values AFTER imputation:
STRATIO       0
PROATCE       0
EDUSHORT      0
STAFFSHORT    0
CLSIZE        0
SCH_ESCS      0
dtype: int64

Student ESCS missing values BEFORE imputation: 4107
Student ESCS missing values AFTER imputation: 0

Merged dimensions: 121564 students across 14 countries.


,before_merge,after_merge,retained_pct,sample_share_%,senwt_sum
CNT,,,,,
AUT,6151,6151,100.0,5.06,1000.0
BEL,8286,8286,100.0,6.82,1000.0
CZE,8460,8460,100.0,6.96,1000.0
DEU,6116,6116,100.0,5.03,1000.0
ESP,30800,30800,100.0,25.34,1000.0
EST,6392,6392,100.0,5.26,1000.0
FIN,10239,10239,100.0,8.42,1000.0
FRA,6770,6770,100.0,5.57,1000.0
ITA,10552,10552,100.0,8.68,1000.0


In [5]:
# Verify completeness of core variables required for Econometrics and ML
core_check_cols = [
    'W_FSTUWT', 'SENWT', 'ESCS', 'SCH_ESCS',
    'PROATCE', 'EDUSHORT', 'CLSIZE', 'STRATIO', 'SCHLTYPE'
] + pv_cols + rep_weight_cols

core_check_cols = [c for c in core_check_cols if c in merged_df.columns]

rows_before = len(merged_df)
merged_df = merged_df.dropna(subset=core_check_cols).copy().reset_index(drop=True)

# Recalculate SENWT after final row filtering so each country strictly sums to 1000.0
merged_df['SENWT'] = merged_df.groupby('CNT')['W_FSTUWT'].transform(
    lambda w: w / w.sum() * 1000.0
)

# Cast SCHLTYPE as categorical for downstream modeling
if 'SCHLTYPE' in merged_df.columns:
    merged_df['SCHLTYPE'] = merged_df['SCHLTYPE'].astype('category')

# Create an explicit composite school group ID for leak-free GroupShuffleSplit in Machine Learning
merged_df['SCHOOL_GROUP_ID'] = merged_df['CNT'].astype(str) + "_" + merged_df['CNTSCHID'].astype(int).astype(str)

print(f"Rows before final cleaning: {rows_before}")
print(f"Rows after final cleaning:  {len(merged_df)}")
print(f"Countries retained in final dataset: {merged_df['CNT'].nunique()} ({sorted(merged_df['CNT'].unique())})")
print(f"Total remaining NaNs across entire dataset: {merged_df.isna().sum().sum()}")

assert merged_df.isna().sum().sum() == 0, "Residual missing values detected!"
assert merged_df['CNT'].nunique() == 14, "One or more countries were accidentally dropped!"
print("\nDataset is 100% clean and verified for Econometrics (BRR + 10 PVs) and ML (GroupSplit).")

Rows before final cleaning: 121564
Rows after final cleaning:  121564
Countries retained in final dataset: 14 (['AUT', 'BEL', 'CZE', 'DEU', 'ESP', 'EST', 'FIN', 'FRA', 'ITA', 'NLD', 'POL', 'PRT', 'QUR', 'SWE'])
Total remaining NaNs across entire dataset: 0

Dataset is 100% clean and verified for Econometrics (BRR + 10 PVs) and ML (GroupSplit).


In [6]:
# Create destination directory for processed analytical data
output_directory = DATA_DIR / "processed"
output_directory.mkdir(parents=True, exist_ok=True)

processed_file_path = output_directory / "pisa_cleaned_dataset.parquet"

# Export to Parquet format (preserves categorical dtypes and compresses 80 replicate weights)
merged_df.to_parquet(processed_file_path, index=False)

print("Pipeline executed successfully.")
print(f"Cleaned dataset saved at: {processed_file_path}")
print(f"Final shape: {merged_df.shape[0]} rows by {merged_df.shape[1]} columns.")
merged_df[['CNT', 'CNTSCHID', 'SCHOOL_GROUP_ID', 'ESCS', 'ESCS_imputed', 'SCH_ESCS', 'W_FSTUWT', 'SENWT', 'SCHLTYPE', 'PV1MATH']].head()

Pipeline executed successfully.
Cleaned dataset saved at: C:\Users\ASUS\pisa_project\data\processed\pisa_cleaned_dataset.parquet
Final shape: 121564 rows by 111 columns.


,CNT,CNTSCHID,SCHOOL_GROUP_ID,ESCS,ESCS_imputed,SCH_ESCS,W_FSTUWT,SENWT,SCHLTYPE,PV1MATH
0,AUT,4000006.0,AUT_4000006,0.1386,0,-0.398280,8.08512,0.106169,3.0,681.761
1,AUT,4000255.0,AUT_4000255,0.6197,0,0.525126,7.16073,0.094031,3.0,513.720
2,AUT,4000267.0,AUT_4000267,1.0992,0,0.929796,18.52873,0.243309,3.0,427.529
3,AUT,4000035.0,AUT_4000035,0.1737,0,0.203346,11.84409,0.155530,3.0,518.294
4,AUT,4000134.0,AUT_4000134,-1.9012,0,-0.069562,7.35583,0.096593,3.0,434.353
